## The full journey: From a corpus to its graph
![](img/processing_chain.jpg)

### Get prepared

#### Install moduls for lemmatization and vectorization

`python -m pip install -U pip setuptools wheel`   
`python -m pip install -U spacy`   
`python -m spacy download en_core_web_sm`

#### Imports for text cleaning, lemmatization, and vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter 
import pandas as pd
pd.options.display.max_rows = 600
import spacy
from bs4 import BeautifulSoup 
import glob
from pathlib import Path
import re
import json
import networkx as nx
from networkx.algorithms import bipartite
import matplotlib.pyplot as plt
import math
import subprocess

#### Read stopwords into `stopw` variable

In [ ]:
file = 'stopwords/stopwords_en_nl'
with open(file, 'r', encoding='utf-8') as stop:
    stopw = stop.read()
stopw = stopw.split('\n')
#stopw

#### Load Spacy language model for english for lemmatization

In [ ]:
nlp = spacy.load("en_core_web_sm")

---

### US Presidential Inaugural Addresses

During Barack Obama’s Inaugural Address in January 2009, he mentioned “women” four different times. We may ask: How distinctive is Obama’s inclusion of women in this address compared to all other U.S. 
Presidents? 

We proceed in four steps:

1. Clean and lemmatize texts in corpus
2. Vectorize the corpus yielding dfidf scores for terms in the graph
3. Construct graph from terms having dfidf scores above given threshold
4. Make SVG file for the graph 

#### Clean and lemmatize

![](img/Lemmatization.jpg)

In [ ]:
directory_path = "./datasets/US_Pres_Inaug_Addr/"
text_files = glob.glob(f"{directory_path}/*.txt")
text_titles = [Path(text).stem for text in text_files]
addresses = []
add_concat = []
xx = []
yy = []
for text_file in text_files:
    with open(text_file, 'r', encoding='utf-8') as file:
        content = file.read()
        content = content.lower()
        content = content.split('\t')[2].replace('\n',' ')
        soup = BeautifulSoup(content, 'html.parser')
        text_no_tags = soup.get_text()
        #print(f"\nLength before stopword removal and lemmatization: {len(text_no_tags)} for {Path(text_file).stem}")
        xx.append(len(text_no_tags))
        doc = nlp(text_no_tags)
        text_lemmatized_list = [token.lemma_ for token in doc if token.text not in stopw and not token.is_punct and not token.lemma_ in stopw]
        text_lemmatized = ' '.join(text_lemmatized_list)
        text = re.sub(r'\$?\s*\d+[,\d+]+', '', text_lemmatized)
        #print (text)
        #print(f"Length after stopw removal and lemmatization:  {len(text)}")
        yy.append(len(text))
        addresses.append(text)
        file.close()
txt_reduct_df = pd.DataFrame({'address' : text_titles, 'before': xx, 'after': yy})
num_nodes_to_inspect = 40


In [ ]:
txt_reduct_df[:num_nodes_to_inspect].plot(x='address', y=['before', 'after'], color=['SlateGray', 'Crimson'], figsize=(15,5), kind='bar')
# compute term frequency dict
addr_concat = ' '.join(addresses)
zz = dict(Counter(addr_concat.split()))
zz = dict(sorted(zz.items()))
term_frq = {k:v for k,v in zz.items()}
#list(term_frq.items())[:15] 

---

### Vectorize

![](img/bag-of-words-vector.webp)

#### Apply tf-idf

In [ ]:
# Fit and transform the presidential addresses using TF-IDF
tfidf_vectorizer = TfidfVectorizer(input='content')
tfidf_vector = tfidf_vectorizer.fit_transform(addresses)
# Make a DataFrame out of the resulting tf–idf vector, setting the "feature names" (terms) as columns and the address titles (i.e. file names) as rows
tfidf_df = pd.DataFrame(tfidf_vector.toarray(), index=text_titles, columns=tfidf_vectorizer.get_feature_names_out())
print ('Terms in dataframe: ',len(tfidf_df.columns))
# Display the DataFrame
tfidf_slice = tfidf_df[['government', 'border', 'people', 'obama', 'war', 'honor','foreign', 'man', 'woman', 'child']]
tfidf_slice.sort_index().round(decimals=2).tail(15)

#### Get document frequency for terms

In [ ]:
doc_frq = (tfidf_df > 0).sum().to_dict()
doc_frq = {k: v for k, v in doc_frq.items() if v > 0}
list(doc_frq.items())[:15]

---

### Create graph

![](img/Strichfiguren.jpeg)

#### Reframe tfidf dataframe

In [ ]:
# Let's reframe the tfidf dataFrame so that the terms are in rows rather than columns.
df_stacked = tfidf_df.stack().reset_index().rename(columns={0:'tfidf', 'level_0': 'address','level_1': 'term'})
print (df_stacked.tail(50))

In [ ]:
seq = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17]
df_stacked.plot(kind='hist', column = 'tfidf', bins=seq, xlabel='tf-idf values')

In [ ]:
seq = [0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18]
df_stacked.plot(kind='hist', column='tfidf', bins=seq,  xlabel='tf-idf values')

#### Set threshold for tf-idf score

In [ ]:
threshold = float(input('Set threshold for tf-idf score: '))

#### Accept terms only, if their tfidf value is greater than chosen threshold

In [ ]:
print ('count rows in full dataframe: ', len(df_stacked.index))
# Set threshold for TF-IDF values and determine remaining terms
thres_tfidf = df_stacked[df_stacked['tfidf'] >= threshold]
print ('count rows in reduced dataframe: ', len(thres_tfidf.index))
thres_tfidf

#### Assign presidential address ID (aka file name) to terms
needed for viz

In [ ]:
used_in = {}
for term in thres_tfidf['term']:
    ad = list(thres_tfidf[thres_tfidf['term'] == term]['address'])
    d = {term: ad}
    used_in.update(d)

#### Create dict for setting node 'id' attribute
needed for viz

In [ ]:
ids = thres_tfidf['term'].unique().tolist()
ids_dict = {k:v for k, v in zip(ids, ids)}
#print (ids_dict)

#### Create graph by projection

In [ ]:
# Now, convert the data frame to a graph by projecting the bipartite graph
B = nx.Graph(created_by='fruschtique')
X = nx.Graph(created_by='fruschtique')
top    = thres_tfidf['address'].unique()
#print (top)
bottom = thres_tfidf['term'].unique()
#print (bottom)
e_list = []
for index, row in thres_tfidf.iterrows():
    e_list.append((row['address'], row['term']))
B.add_nodes_from(top, bipartite=0)
B.add_nodes_from(bottom, bipartite=1)
B.add_edges_from(e_list)
X = bipartite.weighted_projected_graph(B, bottom)
print ('Node count:    ', len(X.nodes()))
print ('Edge count:    ', len(X.edges()))
print ('Graph density: ', nx.density(X))

#### Set node attributes

In [ ]:
nx.set_node_attributes(X, ids_dict, 'id')
nx.set_node_attributes(X, term_frq, 'term_frq')
nx.set_node_attributes(X, doc_frq, 'doc_frq') 
nx.set_node_attributes(X, dict(X.degree()), 'degree')
nx.set_node_attributes(X, dict(nx.betweenness_centrality(X)), 'betweenness')
#print(X.nodes(data=True)) 

As for edge attributes: 
Edge attribute 'weight' is set when projecting the bipartite graph. We don't need any further edge attributes at the moment.

#### Inspect graph attributes

document frequency

In [ ]:
x = {'node': list(doc_frq.keys()), 'doc_frq':list(doc_frq.values())}
doc_frq_df = pd.DataFrame(data=x)
doc_frq_df = doc_frq_df.sort_values(by='doc_frq', ascending=False)
print (doc_frq_df)
num_nodes_to_inspect = 15
doc_frq_df[:num_nodes_to_inspect].plot(x='node', y='doc_frq', color='teal', kind='barh', figsize=[9,6], grid=True).invert_yaxis()

term frequency

In [ ]:
x = {'node': list(term_frq.keys()), 'term_frq':list(term_frq.values())}
term_frq_df = pd.DataFrame(data=x)
term_frq_df = term_frq_df.sort_values(by='term_frq', ascending=False)
print (" Min term_frq: ", term_frq_df["term_frq"].min(), '\n', "Max term_frq: ", term_frq_df["term_frq"].max())
num_nodes_to_inspect = 15
term_frq_df[:num_nodes_to_inspect].plot(x='node', y='term_frq', color='cadetblue', kind='barh', grid=True).invert_yaxis()

degree

In [ ]:
#print (list(X.degree()))
x = {'node': list(nx.get_node_attributes(X, "degree").keys()), 'degree':list(nx.get_node_attributes(X, "degree").values())}
degree_df = pd.DataFrame(data=x)
#print (degree_df)
degree_df = degree_df.sort_values(by='degree', ascending=False)
num_nodes_to_inspect = 15
degree_df[:num_nodes_to_inspect].plot(x='node', y='degree', color='coral', kind='barh', grid=True).invert_yaxis()

betweenness

In [ ]:
x = {'node': list(nx.get_node_attributes(X, "betweenness").keys()), 'betweenness':list(nx.get_node_attributes(X, "betweenness").values())}
betweenness_df = pd.DataFrame(data=x)
betweenness_df = betweenness_df.sort_values(by='betweenness', ascending=False)
num_nodes_to_inspect = 15
betweenness_df[:num_nodes_to_inspect].plot(x='node', y='betweenness', color='darkorange', kind='barh', grid=True).invert_yaxis()

---

### Visualization

![](img/graph.jpg)

#### Required installations

graphviz

#### Initial viz with nx.draw

In [ ]:
# Viz initial graph
pos = nx.spring_layout(X)
plt.figure(figsize=(18, 18))
nx.draw(X, pos, with_labels=True)   
plt.show()

#### Node and edge styling in DOT file
Functions to be called when creating DOT file

In [ ]:
# Functions to compute node attributes

dark = False
directory_path_in = "./misc/"
with open(f"{directory_path_in}meta.json", 'r', encoding='utf-8') as f:
    meta = json.load(f)
    f.close()
#print(meta)

node_coloring = "node coloring: party"
#node_coloring = "node coloring: year"

def computeNodeFillcolor_party(att):
    global dark
    xx = used_in[att['id']]
    y = [meta[x]['party'] for x in xx]
    z = dict(Counter(y))
    if 'Republikaner' in z and 'Demokrat' in z:
        return 'Thistle'
    elif 'Republikaner' in z:
        dark = True
        return '#E9141D'
    elif 'Demokrat' in z:
        dark = True
        return '#0015BC'
    else:
        return 'lightblue'

def computeNodeFillcolor_year(att):
    global dark
    xx = used_in[att['id']]
    ix = xx[0].rfind('_')
    year = xx[0][ix+1:]
    #print(year)
    blue2yellow = [('#081d58ff','dark'), ('#253494ff', 'dark'), ('#225ea8ff', 'dark'), ('#1d91c0ff', 'light'), \
                   ('#41b6c4ff', 'light'), ('#7fcdbbff', 'light'), ('#c7e9b4ff', 'light'), ('#edf8b1ff', 'light'), ('#ffffd9ff', 'light')]
    pastell   = [('#fbb4aeff', 'light'), ('#b3cde3ff', 'light'), ('#ccebc5ff', 'light'), ('#decbe4ff', 'light'), \
                 ('#fed9a6ff', 'light'), ('#ffffccff', 'light'), ('#e5d8bdff', 'light'), ('#fddaecff', 'light'), ('#f2f2f2ff', 'light')]
    palette = blue2yellow
    if year == '1789' or year == '1793':  # G. Washington's addresses
        if palette[0][1] == "dark":
            dark = True
        return palette[0][0]
    elif year >  '1793' and year < '1805':
        if palette[1][1] == "dark":
            dark = True
        return palette[1][0]
    elif year >= '1805' and year < '1833':
        if palette[2][1] == "dark":
            dark = True
        return palette[2][0]
    elif year >= '1833' and year < '1866':
        if palette[3][1] == "dark":
            dark = True
        return palette[3][0]
    elif year >= '1866' and year < '1905':
        if palette[4][1] == "dark":
            dark = True
        return palette[4][0]
    elif year >= '1905' and year < '1933':
        if palette[5][1] == "dark":
            dark = True
        return palette[5][0]
    elif year >= '1933' and year < '1965':
        if palette[6][1] == "dark":
            dark = True
        return palette[6][0]
    elif year >= '1965' and year < '2001':
        if palette[7][1] == "dark":
            dark = True
        return palette[7][0]
    elif year >= '2001':
        if palette[8][1] == "dark":
            dark = True
        return palette[8][0]
    else:
        return 'lightblue'
    
def computeNodeFillcolor(att):
    if node_coloring == "node coloring: party":
        return computeNodeFillcolor_party(att)
    else:
        return computeNodeFillcolor_year(att)
    
def computeNodeWidth(att):
    #print(att.get('term_frq'), att.get('id'))
    w  = 1.0 + math.sqrt(0.2*(att.get('term_frq')-1))
    return w

def computeNodeFontcolor():
    global dark
    if dark:
        dark = False
        return 'white'
    else:
        return 'black'
    
def computeNodeTooltip(att):
    xx = str(used_in[att['id']])
    return xx.replace("[", "").replace("]", "").replace(", ", "\n")

# Functions to compute edge attributes

def computeEdgePenwidth(att):
    return str(att.get('weight')) 

def computeEdgeColor(att):
    if att.get('weight') > 1:
        return 'red'
    else:
        return 'black'

#### Creating DOT file for NetworkX graph (function)

In [ ]:
def graphToDot(graph=None, outfile=None):

   # check for invocation errors
   if graph == None:
      print ('Missing graph specification.')
      return
   if outfile == None:
      print ('Missing output file specification.')
      return
   
   lbl_f_string = f'''   label=<<FONT POINT-SIZE="80">Keywords of the US Presidential Inauguration Addresses 1789 - 2017</FONT><BR ALIGN="LEFT"/><BR ALIGN="LEFT"/>
<FONT POINT-SIZE="60">keyword tf-idf score &gt;= {threshold}, {len(X.nodes())} nodes, {len(X.edges())} edges, graph density: {round(nx.density(X),3)*100}%, {node_coloring}</FONT><BR ALIGN="LEFT"/>>'''

   #lbl_f_string = "This is a test \lof creating my lines\lin a non-centeredway\l"
   
   #dot file header
   dot  = 'graph {\n   overlap="prism1000"\n   rankdir="LR"\n   outputorder="edgesfirst"\n'
   dot += '   fontsize="80"\n   fontname="Arial"\n   labelloc="t"\n   labeljust="l"'
   dot +=     lbl_f_string
   dot += '   node [margin=0 fontname="Arial" fontcolor="black" fontsize=64 shape=circle style=filled];\n'

   # edges
   for u,v,att in graph.edges(data=True):
      dot += f'   {u} -- {v} [id="{u}--{v}"'
      dot += f' penwidth={computeEdgePenwidth(att)}'
      dot += f' color="{computeEdgeColor(att)}"]\n'

   #nodes
   for u,att in graph.nodes(data=True):
      dot += f'   {u} [id="{u}"'
      dot += f' fillcolor="{computeNodeFillcolor(att)}"'  
      dot += f' fontcolor="{computeNodeFontcolor()}"' 
      dot += f' width={computeNodeWidth(att)}' 
      dot += f' tooltip="{computeNodeTooltip(att)}"]\n'

   # close dot file
   dot += '}'

   with open(outfile, 'w', encoding = 'utf8') as file:
      file.write(dot)
   return

#### Node width formula

In [ ]:
xy = {"x": range (1,655), "y": [1.0 + math.sqrt(0.2*(i-1)) for i in range(1,655)]}
node_width = pd.DataFrame(data=xy)
node_width
node_width.plot(kind="scatter", x="x", y="y", figsize=(18,6), xlabel="term_freq value", ylabel="node width factor", grid=True)

#### Create SVG file from DOT file

In [ ]:
# Generate SVG graph and save it
dot_file = f'graphs/USPresInaugAddr_{threshold}.dot'
svg_file = f'graphs/USPresInaugAddr_{threshold}.svg'
outfile = dot_file
graphToDot(X, outfile)
infile  = outfile
print ('in ', infile)
outfile = svg_file
print ('out', outfile)
sfdp = "C:/Program Files/Graphviz/bin/sfdp.exe"
subprocess.run ([f"{sfdp}", f"{infile}", '-o', f"{outfile}", '-Tsvg'])

#### Create HTML embedding for SVG file

In [ ]:
w = 2048
h = 1536
graph_name = f"USPresInaugAddr_{threshold}"

html =  "<!DOCTYPE html>\n  <html>\n    <head>\n    </head>\n    <body>\n      <div>\n"
html += f"        <object data='{graph_name}.svg'  max-width='{w}px'  height='{h}px'>\n"
html += "      </div>\n    </body>\n</html>"
print (html)
with open(f"graphs/{graph_name}.html", "w") as f:
  f.write(html)